In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the dataset
df = pd.read_csv("../data/raw/CO2 Emissions_Canada.csv")

print("Original dataset shape:", df.shape)

Original dataset shape: (7385, 12)


In [3]:
# Create working copy
data = df.copy()

In [4]:
# Rename the Columns
data.rename(columns={ 'Make': 'make',
                    'Model': 'model',
                    'Vehicle Class': 'vehicle_class',
                    'Engine Size(L)': 'engine_size',
                    'Cylinders': 'cylinders',
                    'Transmission': 'transmission',
                    'Fuel Type': 'fuel_type',
                    'Fuel Consumption City (L/100 km)': 'fuel_cons_city',
                    'Fuel Consumption Hwy (L/100 km)': 'fuel_cons_hwy',
                    'Fuel Consumption Comb (L/100 km)': 'fuel_cons_comb',
                    'Fuel Consumption Comb (mpg)': 'fuel_cons_comb_mpg',
                    'CO2 Emissions(g/km)': 'co2'
                    }, inplace=True)

In [5]:
# Validate numeric types for key columns
for col in ['engine_size', 'cylinders', 'co2']:
    data[col] = pd.to_numeric(data[col], errors='coerce')
    print(f"{col} dtype: {data[col].dtype}, nulls: {data[col].isnull().sum()}")

engine_size dtype: float64, nulls: 0
cylinders dtype: int64, nulls: 0
co2 dtype: int64, nulls: 0


In [6]:
# Check missing values
print(data.isnull().sum())

make                  0
model                 0
vehicle_class         0
engine_size           0
cylinders             0
transmission          0
fuel_type             0
fuel_cons_city        0
fuel_cons_hwy         0
fuel_cons_comb        0
fuel_cons_comb_mpg    0
co2                   0
dtype: int64


In [7]:
# Define target
target = "co2"

In [8]:
# Identify leakage variables
leakage_columns = [
    "fuel_cons_city",
    "fuel_cons_hwy",
    "fuel_cons_comb",
    "fuel_cons_comb_mpg"
]

# Checking these are existing in the dataset
for column in leakage_columns:
    print(column, "→", column in data.columns)

fuel_cons_city → True
fuel_cons_hwy → True
fuel_cons_comb → True
fuel_cons_comb_mpg → True


In [9]:
# Remove leakage variables
data_clean = data.drop(columns=leakage_columns).copy()

print("Columns after leakage removal:")
print(data_clean.columns.tolist())
# data_clean = data.copy()  # Keeping all columns for now, but you can uncomment the above lines to remove leakage variables

Columns after leakage removal:
['make', 'model', 'vehicle_class', 'engine_size', 'cylinders', 'transmission', 'fuel_type', 'co2']


In [10]:
# Extract transmission gear count
data_clean["transmission_gears"] = (
    data_clean["transmission"]
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

In [11]:
# Create transmission type
data_clean["transmission_type"] = (
    data_clean["transmission"]
    .str.extract(r"([A-Z]+)", expand=False)
)

In [12]:
# Check transmission extraction
print(
    data_clean[
        ["transmission", "transmission_type", "transmission_gears"]
    ].head(20)
)

print(
    "Missing transmission gears:",
    data_clean["transmission_gears"].isnull().sum()
)

   transmission transmission_type  transmission_gears
0           AS5                AS                 5.0
1            M6                 M                 6.0
2           AV7                AV                 7.0
3           AS6                AS                 6.0
4           AS6                AS                 6.0
5           AS6                AS                 6.0
6           AS6                AS                 6.0
7           AS6                AS                 6.0
8            M6                 M                 6.0
9           AS5                AS                 5.0
10           M6                 M                 6.0
11          AS5                AS                 5.0
12          AM6                AM                 6.0
13           A6                 A                 6.0
14           A6                 A                 6.0
15          AM7                AM                 7.0
16           M6                 M                 6.0
17          AM7             

In [13]:
# Checking whether extraction worked for everything
data_clean["transmission_gears"].isnull().sum()

np.int64(295)

In [14]:
# Handle AV transmissions with no fixed gear count
# Median imputation is used to enable K-Means clustering
median_gears = data_clean["transmission_gears"].median()
data_clean["transmission_gears"] = data_clean["transmission_gears"].fillna(median_gears)

print(f"AV transmissions imputed with median gear count: {median_gears}")
print("Missing transmission_gears after imputation:", data_clean["transmission_gears"].isnull().sum())

AV transmissions imputed with median gear count: 6.0
Missing transmission_gears after imputation: 0


In [15]:
# Engineer interaction features
# f1 = engine_size / cylinders  (displacement per cylinder)
# f2 = engine_size * cylinders  (proxy for combustion chamber volume)
# f3 = engine_size^2            (nonlinear engine size effect)
# f4 = cylinders^2              (nonlinear cylinder effect)

data_clean["engine_per_cylinder"]   = data_clean["engine_size"] / data_clean["cylinders"]       # f1
data_clean["engine_x_cylinders"]    = data_clean["engine_size"] * data_clean["cylinders"]       # f2
data_clean["engine_size_sq"]        = data_clean["engine_size"] ** 2                            # f3
data_clean["cylinders_sq"]          = data_clean["cylinders"] ** 2                              # f4

print("Engineered features added:")
print(data_clean[["engine_per_cylinder", "engine_x_cylinders",
                   "engine_size_sq", "cylinders_sq"]].describe())

Engineered features added:
       engine_per_cylinder  engine_x_cylinders  engine_size_sq  cylinders_sq
count          7385.000000         7385.000000     7385.000000   7385.000000
mean              0.548723           20.040284       11.819557     34.870819
std               0.093753           15.355905       10.243022     24.765008
min               0.250000            2.700000        0.810000      9.000000
25%               0.500000            8.000000        4.000000     16.000000
50%               0.533333           18.000000        9.000000     36.000000
75%               0.600000           22.200000       13.690000     36.000000
max               0.850000          128.000000       70.560000    256.000000


In [16]:
# Remove duplicates — after leakage removal and feature engineering
# This correctly removes rows that are identical in the leakage-free feature space
rows_before_dedup = len(data_clean)

data_clean = data_clean.drop_duplicates().copy()

rows_after_dedup = len(data_clean)

print("Rows before deduplication:", rows_before_dedup)
print("Rows after deduplication:", rows_after_dedup)
print("Duplicates removed:", rows_before_dedup - rows_after_dedup)
print("Total rows removed from original dataset:", len(df) - rows_after_dedup)

Rows before deduplication: 7385
Rows after deduplication: 5985
Duplicates removed: 1400
Total rows removed from original dataset: 1400


In [17]:
# Check duplicates AFTER feature engineering
duplicates_after_features = data_clean.duplicated().sum()

print(
    "Remaining duplicates after feature engineering and deduplication:",
    duplicates_after_features
)

Remaining duplicates after feature engineering and deduplication: 0


In [18]:
# Define engineering features used for temporal clustering (KMeans)
engineering_features = ["engine_size", "cylinders", "transmission_gears"]

print(data_clean[engineering_features].describe())
print("Any nulls in clustering features:", data_clean[engineering_features].isnull().sum().sum())

       engine_size    cylinders  transmission_gears
count  5985.000000  5985.000000         5985.000000
mean      3.159783     5.616374            6.842607
std       1.367660     1.853022            1.207689
min       0.900000     3.000000            4.000000
25%       2.000000     4.000000            6.000000
50%       3.000000     6.000000            6.000000
75%       3.700000     6.000000            8.000000
max       8.400000    16.000000           10.000000
Any nulls in clustering features: 0


In [19]:
# Final shape and column check
print("Final dataset shape:", data_clean.shape)
print("Columns:", data_clean.columns.tolist())
print("\nMissing values in final dataset:")
print(data_clean.isnull().sum()[data_clean.isnull().sum() > 0])

Final dataset shape: (5985, 14)
Columns: ['make', 'model', 'vehicle_class', 'engine_size', 'cylinders', 'transmission', 'fuel_type', 'co2', 'transmission_gears', 'transmission_type', 'engine_per_cylinder', 'engine_x_cylinders', 'engine_size_sq', 'cylinders_sq']

Missing values in final dataset:
Series([], dtype: int64)


In [20]:
# Save processed data
data_clean.to_csv(
    "../data/processed/vehicle_co2_clean.csv",
    index=False
)